### **다음과 같은 요구사항을 Gradio ChatInterface로 구현합니다**

- 주제: 맞춤형 여행 일정 계획 어시스턴트
- 기능: 
   - OpenAI Chat Completion API와 LangChain을 활용하여 사용자의 선호도에 맞는 여행 일정을 생성
   - LCEL을 사용하여 단계별 프롬프트 체인 구성 (사용자 입력 분석 -> 일정 생성 -> 세부 계획 수립)
   - 채팅 히스토리 사용하여 답변 생성
   - Gradio 인터페이스를 통해 사용자와 대화형으로 상호작용

- 주요 포인트:

   1. **모델 매개변수 최적화**
      - temperature=0.7: 적당한 창의성을 유지하면서 일관된 응답 생성
      - top_p=0.9: 높은 확률의 토큰만 선택하여 응답의 품질 향상
      - presence_penalty와 frequency_penalty: 반복적인 응답을 줄이고 다양한 제안 생성

   2. **시스템 프롬프트 설계**
      - 여행 플래너로서의 역할과 응답 가이드라인을 명확히 정의
      - 구체적인 정보를 포함하도록 지시
      - 한국어 응답 명시

   3. **메모리 관리**
      - Gradio 또는 LangChain 메모리 기능을 사용하여 대화 컨텍스트 유지
      - 이전 대화 내용을 바탕으로 연속성 있는 응답 생성

### 단계별 구현 가이드

- **Step 1**: 시스템 프롬프트 작성
   - 여행 플래너의 역할 명시
   - 구체적인 정보 포함 지시 (날짜, 장소, 예산 등)

- **Step 2**: 채팅 히스토리 관리
   - MessagesPlaceholder 사용
   - HumanMessage/AIMessage 변환

- **Step 3**: Gradio 인터페이스 설정
   - 제목과 설명 추가

**Step 4**: 테스트
   - "서울에서 2박 3일 여행 계획 짜줘" 등의 질문 시도

In [5]:
# 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

# LLM 모델 정의
model = ChatOpenAI(
    model="gpt-4.1-nano", 
    temperature=0.7, 
    top_p=0.9,
    presence_penalty=0.3,
    frequency_penalty=0.3,
    )

# 메시지 플레이스홀더가 있는 프롬프트 템플릿 정의
analyze_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 여행 일정을 계획해주는 여행 전문가 입니다. 사용자의 요청과 이전 대화 내용을 바탕으로, 
    여행 계절, 선호 스타일(휴양, 액티비티, 맛집), 여행 지역으로 요약하세요.

    1. 여행 계절 ex) 겨울
    2. 선호 스타일 ex) 휴양
    3. 여행 지역 ex) 서울, 강원도
    """
    ), 
    MessagesPlaceholder("chat_history"),
    ("human", "{user_input}")
])

make_schedule_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    사용자의 성향과 요청 내용을 가지고, 대략적인 일정, 지역 방문 순서, 여행지 핵심 특성을 가지고 제안해주세요.

    # 사용자의 성향 
    {analysis_result}
    """
    ), 
    ("human", "내 성향에 맞춰서 맞춤형 일정을 제안해줘")
])

plan_detail_prompt =  ChatPromptTemplate.from_messages([
    ("system", """
    제안된 일정을 가지고, 세부적인 계획을 수립해줘 여행할때 필요한 준비물도 표시해줘

    # 제안된 일정 
    {make_schedule_result}
    """
    ), 
    ("human", "세부 계획과 준비물은 무엇이 필요한지 알려줘")
])

# LCEL을 사용하여 단계별 프롬프트 체인 구성 (사용자 입력 분석 -> 일정 생성 -> 세부 계획 수립)

analyze_chain = analyze_prompt | model | StrOutputParser()
schedule_chain = make_schedule_prompt | model | StrOutputParser()
detail_chain = plan_detail_prompt | model |  StrOutputParser()

chain = (
    {"analysis_result": analyze_chain}  
    | RunnablePassthrough.assign(make_schedule_result=schedule_chain)
    | detail_chain 
)

def answer_invoke(message, history):

    history_messages = []
    for msg in history:
        if msg['role'] == "user":
            history_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == "assistant":
            history_messages.append(AIMessage(content=msg['content']))

    history_messages.append(HumanMessage(content=message))
    response = chain.invoke({
        "chat_history": history_messages,
        "user_input": message
    })
    return response

# Gradio ChatInterface 객체 생성
demo = gr.ChatInterface(
    fn=answer_invoke,         # 메시지 처리 함수
    title="맞춤형 여행 일정 계획 어시스턴트", # 채팅 인터페이스의 제목
    )

# Gradio 인터페이스 실행
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [6]:
# Gradio 인터페이스 종료
demo.close()

Closing server running on port: 7864


### 입력 
부산 1박 2일 6월

### 결과
물론입니다! 제안드린 부산 여행 일정에 맞춘 세부 계획과 필요한 준비물을 상세히 안내해드릴게요.

1일차: 해운대와 광안리 해변, 맛집 탐방
세부 계획
오전
08:00~09:00: 숙소에서 출발, 해운대 해수욕장 도착
09:00~10:30: 해변 산책 및 동백섬, APEC 나루공원 방문
점심
11:00~12:30: 해운대 맛집 또는 시장 내 신선한 해산물 즐기기
오후
13:00~15:00: 센텀시티 쇼핑몰 또는 벡스코 전시장 방문
저녁
16:00~18:00: 광안리 해변으로 이동, 야경 감상
18:30 이후: 광안리 수제맥주 거리 또는 맛집에서 저녁 식사 후 자유시간
2일차: 도심과 문화 체험
세부 계획
오전
09:00~10:30: 자갈치시장 방문, 싱싱한 해산물 구경 및 간단한 먹거리 체험
11:00~12:30: 부산 국제시장과 깡통시장 탐방
점심
13:00~14:00: 시장 내 맛집 또는 길거리 음식 체험 (돼지국밥, 밀면)
오후
14:30~16:00: 감천문화마을 산책, 색감 넘치는 벽화와 마을 구경
16:30~17:30: 용두산공원과 부산타워 방문하여 시내 전경 감상
저녁
18:00 이후 : 밀면 또는 꼼장어구이 집에서 저녁식사 후 휴식
3일차 : 자연과 휴양
세부 계획
오전
09:00~10:30 : 송도해수욕장 또는 다대포해수욕장 방문, 여유로운 시간 보내기
점심
11:00~12:30 : 바닷가 근처 카페 또는 레스토랑에서 가볍게 식사
오후
13:00~15:00 : 태종대 또는 오륙도 전망대 방문, 멋진 바다 풍경 감상 및 산책
저녁
오후 늦게 귀가 또는 마지막 밤은 다시 광안리·해운대에서 야경 감상하며 마무리
여행 준비물 리스트
기본 준비물
신분증/신용카드 : 신분증(여권 가능), 결제용 카드 필수
모바일 충전기 & 보조배터리 : 스마트폰 사용이 많으니 꼭 챙기세요
여행 일정표 인쇄본 또는 저장된 파일 : 일정 체크용
의류 및 개인용품
가볍고 통풍이 잘 되는 옷 (여름철)
수영복 & 수건 (해변 활동 용)
모자, 선크림, 선글라스 (햇볕 차단)
편한 신발(샌들 또는 운동화) — 도보 많음!
얇은 긴팔 셔츠 혹은 외투 (저녁이나 바람 부는 날 대비)
기타 필요 용품
개인 위생용품 (칫솔, 치약, 샴푸 등)
선크림 & 립밤 (자외선 차단 필수)
물병 & 간단한 간식(야외 활동 시 필요)
방수팩 또는 비닐봉지(젖거나 더러워진 물품 보관용)
선택적 준비물
카메라 또는 액션캠 : 추억 남기기 위해서
여행 가이드북 또는 지도 앱 다운로드
작은 배낭 또는 크로스백 : 소지품 휴대용
이외에도 여행 기간 동안 날씨를 확인하고 그에 맞는 옷차림을 준비하는 것도 중요합니다. 즐거운 부산 여행 되시길 바랍니다!




# LangSmith 
 
## 사용자 입력 분석 Chain
### INPUT 
    SYSTEM : 
    당신은 여행 일정을 계획해주는 여행 전문가 입니다. 사용자의 요청과 이전 대화 내용을 바탕으로, 
    여행 계절, 선호 스타일(휴양, 액티비티, 맛집), 여행 지역으로 요약하세요.

    1. 여행 계절 ex) 겨울
    2. 선호 스타일 ex) 휴양
    3. 여행 지역 ex) 서울, 강원도
    
    USER : 부산 1박 2일 6월

## 일정 생성 Chain 0
### INPUT 
    SYSTEM : 
    사용자의 성향과 요청 내용을 가지고, 대략적인 일정, 지역 방문 순서, 여행지 핵심 특성을 가지고 제안해주세요.

    # 사용자의 성향 
    1. 여행 계절: 여름

    2. 선호 스타일: 해변과 맛집 탐방 중심의 휴양 및 시내 관광
    3. 여행 지역: 부산

    USER : 
    내 성향에 맞춰서 맞춤형 일정을 제안해줘

### OUTPUT 
    물론입니다! 여름철 부산에서 해변과 맛집 탐방을 중심으로 한 휴양 및 시내 관광 일정은 다음과 같이 추천드릴게요.
    1일차: 해운대와 광안리 해변, 맛집 탐방

    오전
    해운대 해수욕장 산책 및 해변 감상
    동백섬과 APEC 나루공원 방문...

## 세부 계획 수립 Chain 

### INPUT 
    SYSTEM : 
    제안된 일정을 가지고, 세부적인 계획을 수립해줘 여행할때 필요한 준비물도 표시해줘

    # 제안된 일정 
    물론입니다! 여름철 부산에서 해변과 맛집 탐방을 중심으로 한 휴양 및 시내 관광 일정은 다음과 같이 추천드릴게요.
    ...
    
    USER : 
    세부 계획과 준비물은 무엇이 필요한지 알려줘




# 추가 개선 방안
"""  
 1.  안내에 필요한 정보가 없을떄 입력 받기
 지역 : 가고싶으신 지역을 선택해주세요.
 시기 : 언제쯤 여행을 희망하시나요?
 선호 여행 : 휴양, 맛집 탐방, 관광 등 선호하는 여행 스타일이 있으신가요?

 2. 실제 여행 후기 찾아주기
    최종 입력 받았을때, 사용률이 높은 여행 사이트들의 실제 여행 리뷰 후기 보여주기 
"""